In [ ]:
# Single-cell verification of the projection / YOLO-decode fixes on dhaneshkk/screen-eye-tracking.
# Runs on a CPU Colab runtime; no models or webcam needed.
import subprocess, sys, os, time, json
BRANCH = "fix/projection-flip-convergence-yolo-decode"
REPO = "/content/screen-eye-tracking"
def sh(cmd): print("$", cmd); subprocess.run(cmd, shell=True, check=True)
if not os.path.isdir(REPO):
    sh(f"git clone --depth 1 -b {BRANCH} https://github.com/dhaneshkk/screen-eye-tracking {REPO}")
sh(f"{sys.executable} -m pip install -q numpy opencv-python-headless onnxruntime pytest")
os.chdir(REPO); sys.path[:0] = [f"{REPO}/src", f"{REPO}/tests"]

# 1) Full Python suite (includes the old-loop equivalence test and shared-fixture check).
sh(f"{sys.executable} -m pytest -q")

# 2) The two geometry bugs, before vs after.
from screen_eye_tracking.backend import ScreenProjector, DisplayGeometry, Detection, GazeEstimate, parse_yolo_output
import math, numpy as np
d = DisplayGeometry(1920, 1080, 31.5)
for cx in (0.0, 0.2, 0.5):
    x = ScreenProjector(d, flip_x=True, camera_screen_x=cx).project((320, 240), 0, 0, 0.6)[0]
    print(f"camera_screen_x={cx}: looking into lens -> x_norm={x:.3f} (upstream gave {1-cx:.3f})")
p = ScreenProjector(d, flip_x=False, flip_y=False); D = 0.6; W = d.size_m[0]; f = p.focal_px
half = 0.0315 * f / D
eyes = [Detection(17, 1, 316-half, 236, 324-half, 244), Detection(17, 1, 316+half, 236, 324+half, 244)]
tx = W/2 + 0.25
yl = math.degrees(math.atan((tx-(W/2-0.0315))/D)); yr = math.degrees(math.atan((tx-(W/2+0.0315))/D))
for e in (-1.5, 0.0, 1.5):
    r = p.project_estimate("binocular-convergence", eyes, GazeEstimate(0, 0, yl+e, 0, yr-e, 0), D)
    print(f"vergence error {e:+.1f} deg -> x_norm={r.point[0]:.3f} (true {tx/W:.3f})")

# 3) YOLO decode timing.
from test_yolo_vectorized import random_output
o = random_output(0, 0.05); t = time.perf_counter()
for _ in range(50): parse_yolo_output(o, 0.5)
print(f"parse_yolo_output: {(time.perf_counter()-t)/50*1000:.2f} ms/frame (upstream ~10 ms)")
